In [ ]:
#!pip install google-generativeai
import configparser
import google.generativeai as genai
import json
import re
from google.colab import userdata
import datetime

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
## References:
## https://chatgpt.com/share/67353362-8e1c-800b-a6eb-f8db73f2160d
##

In [ ]:
# Generate multipe responses in on go and save to json file

# Set up API key and model
API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel(model_name="gemini-1.5-flash")

# Load prompts from configuration
prompts = configparser.ConfigParser()
prompts.read('/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Zang/prompts.env')
my_prompt = prompts.get('TEMPLATES', 'TOPIC')  # Simplified prompt

# Function to clean text and parse it as JSON
def clean_json_text(text):
    # Remove surrounding ```json tags and any unnecessary characters
    cleaned_text = text.strip("`").replace("json", "", 1).strip()
    cleaned_text = re.sub(r'\\n|\\', '', cleaned_text)

    # Attempt to parse cleaned text as JSON
    try:
        parsed_json = json.loads(cleaned_text)
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return None

    return parsed_json

# Function to save response to JSONL, appending with unique ID
def save_response_to_jsonl(response, output_file="output.jsonl"):
    # Clean and parse the JSON content
    cleaned_data = clean_json_text(response["text"])
    if cleaned_data is not None:
        # Prepare the structure for saving without "text" key
        response_data = {
            "id": response["id"],
            "model_input": cleaned_data["model_input"],
            "model_output_text": cleaned_data["model_output_text"],
            "soft_labels": cleaned_data.get("soft_labels", []),
            "hard_labels": cleaned_data.get("hard_labels", [])
        }

        # Save the response to a JSONL file (appending)
        with open(output_file, "a") as jsonl_file:
            jsonl_file.write(json.dumps(response_data) + "\n")
        print(f"Response with ID {response['id']} saved to {output_file}")
    else:
        print(f"Failed to save response with ID {response['id']} due to parsing error.")

# Loop to generate and save multiple responses
def generate_and_save_responses(num_responses=5, output_file="output.jsonl"):
    # Initial ID to start from
    current_id = 10

    # Loop to generate and save 'num_responses' responses
    for _ in range(num_responses):
        # Generate a new response
        response = model.generate_content(my_prompt)
        response_data = {"id": current_id, "text": response.text}  # Include unique ID

        # Save response to JSONL
        save_response_to_jsonl(response_data, output_file)

        # Increment ID for next response
        current_id += 1

path_to_data = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Zang/outputs"

# Get current date in the desired format (e.g., "14-11-2024")
current_date = datetime.datetime.now().strftime("%d-%m-%Y")

# Example usage: Generate 5 responses and save to a file named with the current date
output_file = f"{path_to_data}/output-{current_date}.jsonl"
generate_and_save_responses(num_responses=5, output_file=output_file)

Response with ID 5 saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Zang/outputs/output-13-11-2024.jsonl
Response with ID 6 saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Zang/outputs/output-13-11-2024.jsonl
Response with ID 7 saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Zang/outputs/output-13-11-2024.jsonl
Response with ID 8 saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Zang/outputs/output-13-11-2024.jsonl
Response with ID 9 saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Zang/outputs/output-13-11-2024.jsonl
